# Microbiome 16S rRNA Sequencing Environment Prediction ML Project
Thillai Sudhakar
December 2025

## Data Ingestion
For this project, I used 16S sequencing data from the _Standardized multi-omics of Earths microbiomes reveals microbial and metabolite diversity_ study in [Qiita](https://qiita.ucsd.edu/study/description/13114#). 

In [0]:
%pip install biom-format
%pip install pandas


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# imports
import biom
import pandas as pd
import json

In [0]:
# load files
table = biom.load_table('/Volumes/workspace/default/microbiome_project_files/155185_otu_table.biom')
metadata = pd.read_csv('/Volumes/workspace/default/microbiome_project_files/sample_information_from_prep_9957.tsv', sep='\t')

This taxonomy mapping is useful in later notebooks, when we want to see the taxonomy of significant OTU IDs. It will also be useful to do genus-level aggregation because the OTU-level (31k features) is too sparse, and the genus-level can capture biological signal while being computationally tractable.

In [0]:
# aggregate by lowest classified level
from collections import defaultdict

# get all OTU IDs
otu_ids = table.ids(axis='observation')

# create a mapping of taxonomy_name -> list of OTU IDS
taxonomy_to_otus = defaultdict(list)

# aggregate counts by taxonomy
for otu_id in otu_ids:
    otu_metadata = table.metadata(otu_id, axis='observation')

    if otu_metadata and 'taxonomy' in otu_metadata:
        tax_list = otu_metadata['taxonomy']

        # find the lowest classified level (work backwards from genus)
        genus = tax_list[5] if len(tax_list) > 5 and tax_list[5] != 'g__' else None
        family = tax_list[4] if len(tax_list) > 4 and tax_list[4] != 'f__' else None
        order = tax_list[3] if len(tax_list) > 3 and tax_list[3] != 'o__' else None

        # use the first non-empty level
        tax_name = genus or family or order or 'unclassified'

        taxonomy_to_otus[tax_name].append(otu_id)

print(f"aggregated {len(table.ids(axis='observation'))} OTUs into {len(taxonomy_to_otus)} taxonomic groups")


aggregated 31379 OTUs into 1824 taxonomic groups


In [0]:
# convert biom table to pandas dataframe
otu_df = table.to_dataframe(dense=True)
print(f"otu dataframe shape: {otu_df.shape}")
print(f"first 5 sample ids:\n{otu_df.columns[:5].tolist()}")

otu dataframe shape: (31379, 474)
first 5 sample ids:
['13114.rohwer.84.s004', '13114.control.soil.under.pine.tree.by.Skaggs', '13114.control.soil.under.rock.near.century.plant', '13114.rohwer.84.s002', '13114.mayer.33.s003']


In [0]:
# aggregate otus by taxonomy
aggregated_data = {}

for tax_name, otu_list in taxonomy_to_otus.items():
    # get rows for all otus in this taxonomy group
    otu_rows = otu_df.loc[otu_list]
    # sum across all otus in the group (axis=0 sums down rows)
    aggregated_data[tax_name] = otu_rows.sum(axis=0)

# convert dict to dataframe
otu_df_agg = pd.DataFrame(aggregated_data).T

print(f"aggregated otu shape: {otu_df_agg.shape}")
print(f"reduced from {len(otu_ids)} otus to {len(otu_df_agg)} taxonomy groups")

aggregated otu shape: (1824, 474)
reduced from 31379 otus to 1824 taxonomy groups


In [0]:
# check sample overlap
otu_samples = set(otu_df_agg.columns)
meta_samples = set(metadata['sample_name'])

print(f"samples in otu table: {len(otu_samples)}")
print(f"samples in metadata: {len(meta_samples)}")
print(f"overlap: {len(otu_samples & meta_samples)}")
print(f"only in otu: {len(otu_samples - meta_samples)}")
print(f"only in metadata: {len(meta_samples - otu_samples)}")

samples in otu table: 474
samples in metadata: 480
overlap: 474
only in otu: 0
only in metadata: 6


In [0]:
# filter to only overlapping samples
common_samples = list(otu_samples & meta_samples)
otu_df_filtered = otu_df[common_samples]
metadata_filtered = metadata[metadata['sample_name'].isin(common_samples)]

print(f"filtered to {len(common_samples)} common samples")


filtered to 474 common samples


In [0]:
# transpose otu table so samples are rows (needed for sklearn)
# shape will be (samples, otus)
otu_df_t = otu_df_filtered.T
otu_df_t.index.name = 'sample_name'
otu_df_t = otu_df_t.reset_index()

print(f"transposed otu shape: {otu_df_t.shape}")

transposed otu shape: (474, 31380)


In [0]:
# merge otu data with metadata
merged = otu_df_t.merge(metadata_filtered, on='sample_name', how='inner')
print(f"merged shape: {merged.shape}")

merged shape: (474, 31730)


`empo_3` is ultimately what we'll be trying to predict. Let's take a look at the distribution here.

In [0]:
# check empo_3 distribution
print("environment type distribution:")
print(merged['empo_3'].value_counts())

environment type distribution:
empo_3
Animal distal gut          71
Sediment (saline)          65
Plant surface              49
Sterile water blank        38
Soil (non-saline)          37
Water (saline)             36
Sediment (non-saline)      36
Animal corpus              33
Animal proximal gut        25
Subsurface (non-saline)    23
Water (non-saline)         22
Animal secretion           20
Fungus corpus              12
Single strain               4
Surface (saline)            3
Name: count, dtype: int64


In [0]:
# remove control/blank samples
# keep only actual environmental samples
controls_to_remove = ['Sterile water blank', 'Single strain', 'Mock community']
merged_clean = merged[~merged['empo_3'].isin(controls_to_remove)]

print(f"after removing controls: {merged_clean.shape}")
print(f"\nfinal environment distribution:")
print(merged_clean['empo_3'].value_counts())

after removing controls: (432, 31730)

final environment distribution:
empo_3
Animal distal gut          71
Sediment (saline)          65
Plant surface              49
Soil (non-saline)          37
Water (saline)             36
Sediment (non-saline)      36
Animal corpus              33
Animal proximal gut        25
Subsurface (non-saline)    23
Water (non-saline)         22
Animal secretion           20
Fungus corpus              12
Surface (saline)            3
Name: count, dtype: int64


In [0]:
# save processed data for next step
output_path = '/Volumes/workspace/default/microbiome_project_files/processed/merged_data.parquet'
merged_clean.to_parquet(output_path, index=False)
print(f"saved to {output_path}")

saved to /Volumes/workspace/default/microbiome_project_files/processed/merged_data.parquet


In [0]:
# save taxonomy mapping as json
taxonomy_map_serializable = {str(k): v for k, v in taxonomy_to_otus.items()}

taxonomy_path = '/Volumes/workspace/default/microbiome_project_files/processed/taxonomy_to_otus.json'
with open(taxonomy_path, 'w') as f:
    json.dump(taxonomy_map_serializable, f)

print(f"saved taxonomy mapping for {len(taxonomy_map_serializable)} taxonomic groups")

saved taxonomy mapping for 1824 taxonomic groups


In [0]:
# quick sanity checks
print(f"total samples: {len(merged_clean)}")
print(f"total otus: {len([c for c in merged_clean.columns if c not in metadata_filtered.columns])}")
print(f"number of environment types: {merged_clean['empo_3'].nunique()}")
print(f"any nulls in empo_3: {merged_clean['empo_3'].isnull().any()}")

total samples: 432
total otus: 31379
number of environment types: 13
any nulls in empo_3: False
